# ML-Based Market Regime Detection & Risk Intelligence System
End-to-end walkthrough using the same modules as the Streamlit app (`src/`).

**Important:** regimes are *descriptive* labels of historical conditions found with unsupervised clustering.
Nothing here predicts future prices or gives trading signals.

Steps: 1) download data → 2) engineer features → 3) choose k (K-Means vs GMM) → 4) full pipeline →
5) regimes → 6) risk → 7) evaluation → 8) optional MySQL → 9) launch dashboards.

In [1]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():
    ROOT = ROOT.parent          # notebook lives in notebooks/
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import pandas as pd
from src import config, data_loader, features, regime_models as rm, risk_engine as risk, pipeline, plots
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)

TICKERS = ["NVDA", "AAPL", "MSFT", "AMD", "QQQ"]
YEARS = 5            # use 5-10
N_REGIMES = 3        # 2-6
TRAIN_FRAC = 0.70    # chronological split
CMAP = config.regime_color_map(N_REGIMES)

## 1. Download real daily OHLCV data (yfinance)

In [2]:
prices, errors, warnings_ = data_loader.load_prices(TICKERS, YEARS)
print("Errors  :", errors)
print("Warnings:", warnings_)
print(prices.groupby("ticker")["trade_date"].agg(["min", "max", "count"]))
prices.head()

Errors  : {'NVDA': "Download failed: No module named 'yfinance'", 'AAPL': "Download failed: No module named 'yfinance'", 'MSFT': "Download failed: No module named 'yfinance'", 'AMD': "Download failed: No module named 'yfinance'", 'QQQ': "Download failed: No module named 'yfinance'"}
Warnings: []
Empty DataFrame
Columns: [min, max, count]
Index: []


,ticker,trade_date,open,high,low,close,volume


## 2. Feature engineering
Daily/5-day returns, 20-day rolling volatility, MA ratios, volume change, RSI(14), 60-day max drawdown. All use trailing windows only.

In [3]:
feats = features.build_features(prices)
print(feats.shape); print(config.FEATURE_DESCRIPTIONS)
feats[feats.ticker == TICKERS[0]].tail()

(0, 13)
{'daily_return': '1-day simple return', 'return_5d': '5-day simple return', 'volatility_20d': '20-day rolling std of daily returns, annualised', 'ma20_ratio': 'close / 20-day moving average - 1', 'ma50_ratio': 'close / 50-day moving average - 1', 'volume_change': 'volume / 20-day average volume - 1', 'rsi_14': '14-day RSI (Wilder smoothing)', 'max_drawdown_60d': 'worst drawdown from a rolling peak over 60 days'}


,ticker,trade_date,daily_return,return_5d,volatility_20d,ma_20,ma_50,ma20_ratio,ma50_ratio,volume_change,rsi_14,max_drawdown_60d,drawdown


## 3. Model selection: how many regimes? (K-Means vs GMM)
The scaler and models are fitted on the **first 70% of days only**; the last 30% is an unseen test period.

In [4]:
T = TICKERS[0]
ft = feats[feats.ticker == T].sort_values("trade_date").reset_index(drop=True)
sweep = rm.selection_sweep(T, ft, TRAIN_FRAC)
display(sweep.round(3))
plots.selection_curves(sweep).show()

,ticker,model_name,n_clusters,silhouette_train,silhouette_test,inertia_train,bic,aic


## 4. Run the complete pipeline for every ticker

In [5]:
result = pipeline.run_pipeline(prices=prices, n_regimes=N_REGIMES, train_frac=TRAIN_FRAC)
print(result.meta["tickers"], result.meta["errors"])
{k: v.shape for k, v in result.tables().items()}

PipelineError: No price data could be loaded (no data).

## 5. Regimes: timeline, durations, frequencies, transitions, statistics

In [ ]:
MODEL = "GMM"      # or "KMeans"
T = result.tickers[0]
pred = result.regime_predictions.query("ticker == @T and model_name == @MODEL").sort_values("trade_date")
split_date = pred.loc[pred.dataset_split == "test", "trade_date"].min()
plots.regime_timeline(pred, CMAP, split_date, f"{T} regimes ({MODEL})").show()
plots.price_by_regime(result.prices.query("ticker == @T"), pred, CMAP).show()

In [ ]:
episodes = rm.compute_episodes(pred)
display(rm.duration_summary(episodes).round(1))
display(rm.frequency_table(pred))
counts, probs = rm.transition_matrices(pred, N_REGIMES)
plots.transition_heatmap(probs).show()
ft_t = result.features.query("ticker == @T")
display(rm.cluster_statistics(ft_t, pred).round(4))

In [ ]:
pca = result.pca.query("ticker == @T")
var = result.pca_variance.query("ticker == @T").iloc[0]
plots.pca_scatter(pca, pred, CMAP, (var.pc1_var, var.pc2_var)).show()

## 6. Risk engine
Historical VaR / Expected Shortfall are reported as positive one-day loss fractions.

In [ ]:
display(result.risk_metrics.round(4))
plots.var_es_bar(result.risk_metrics).show()
rets = {t: risk.returns_from_prices(result.prices.query("ticker == @t")) for t in result.tickers}
plots.rolling_vol_fig(rets, 20).show()
plots.drawdown_fig(rets).show()

In [ ]:
rr = result.regime_risk_metrics.query("ticker == @T and model_name == @MODEL")
display(rr.round(4))
plots.regime_risk_bar(rr, CMAP).show()
display(result.high_risk_periods.query("ticker == @T").sort_values("peak_volatility", ascending=False).head(10).round(3))

## 7. Model evaluation

In [ ]:
display(result.model_evaluation.round(3))
plots.stability_bar(result.model_evaluation).show()
from src import insights
for line in insights.generate_insights(result, MODEL):
    print("-", line)

## 8. Optional: store everything in MySQL
Requires a `.env` file (copy `.env.example`) and MySQL 8+. If MySQL is not set up this cell prints the reason and the rest still works.

In [ ]:
from src import database
ok, msg = database.check_connection()
print(msg)
if ok:
    print(database.init_db())
    print(pipeline.save_result_to_db(result))
    eng = database.get_engine()
    display(database.table_counts(eng))
    display(database.run_select(eng, "SELECT ticker, regime_name, COUNT(*) AS days FROM regime_predictions WHERE model_name='GMM' GROUP BY ticker, regime_id, regime_name ORDER BY ticker, regime_id"))

In [ ]:
# Make the Streamlit app open with these results (no re-download needed)
pipeline.save_local(result)
print("Saved. Now run in a terminal:  streamlit run app.py")

## 9. Launch the dashboards
```
streamlit run app.py
```